In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_groq import ChatGroq
from langgraph.graph import END, START
from langgraph.graph.state import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage
import os 
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

In [3]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [4]:
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [5]:
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

tool_node = ToolNode([add])

llm_with_tool = llm.bind_tools([add])

def tool_calling_llm(state: State):

    return {"messages": [llm_with_tool.invoke(state["messages"])]}

In [6]:
from langgraph.prebuilt import tools_condition

builder = StateGraph(State)

builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", tool_node)  # Changed from "tool_node" to "tools"

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)

builder.add_edge("tools", "tool_calling_llm")  # Updated edge reference

graph = builder.compile()

In [7]:
from langchain_core.messages import HumanMessage

response = graph.invoke({"messages": [HumanMessage(content="What is 3 + 5?")]})
print(response)

{'messages': [HumanMessage(content='What is 3 + 5?', additional_kwargs={}, response_metadata={}, id='90c8b86e-39c9-4524-8473-d6d65cbdd1d7'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'fc_17024f6e-6aea-4264-8da2-11cf29740602', 'function': {'arguments': '{"a":3,"b":5}', 'name': 'add'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 128, 'total_tokens': 174, 'completion_time': 0.045167687, 'prompt_time': 0.006155214, 'queue_time': 0.042944816, 'total_time': 0.051322901}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--da2e94d0-43df-48c9-a34d-bf49b0642eea-0', tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': 'fc_17024f6e-6aea-4264-8da2-11cf29740602', 'type': 'tool_call'}], usage_metadata={'input_tokens': 128, 'output_tokens': 46, 'total_tokens': 174}), ToolMessage(content='8', name='add', id='79676c8c-6197-4045-b1b0-310325